# Tutorial: Modelos de Decisión estilo JEV (System 1)

**Para principiantes.** Vamos a entender paso a paso cómo funciona un "modelo de decisión": un modelo que, dado un texto de entrada (un ticket, un email, una petición) y una lista de preguntas tipadas, devuelve **respuestas con probabilidades calibradas en una sola pasada** — sin generar texto libre.

**Referencias:**
- JEV Decision Index (leaderboard de modelos de decisión): https://huggingface.co/spaces/multimodalart/jev-decision-index
- Laya (modelo open source estilo JEV que usaremos al final): https://huggingface.co/convaiinnovations/laya

**Plan del tutorial (3 pasos):**

| Paso | Técnica | Idea clave |
|------|---------|-----------|
| 1 | LLM generativo (Qwen2.5 / SmolLM2) | Leer los logits del *siguiente token* y convertirlos en porcentajes |
| 2 | Encoder zero-shot (DeBERTa NLI) | Clasificar comparando el texto con hipótesis, sin prompt |
| 3 | Laya (modelo de decisión estilo JEV) | Un solo forward pass para TODAS las preguntas tipadas: `choice`, `score`, `noul` |

---

## ¿Qué es un modelo de decisión "estilo JEV"?

Los LLM tradicionales son **autoregresivos (System 2)**: generan texto token a token, lo que es lento y además produce texto libre que hay que *parsear* (y puede alucinar).

Los modelos de decisión (Jev, Laya, Bosun, Decider...) son **System 1**: modelos que responden **preguntas tipadas** sobre un estado de entrada:

- **`choice`** → elegir entre opciones ("¿qué departamento?") → devuelve una etiqueta + distribución de probabilidad
- **`score`** → puntuación ordinal ("¿cuán urgente es?") → devuelve un valor tipo `1.84 / 2.0`
- **`noul`** → pregunta sí/no → devuelve una **probabilidad** de que la respuesta sea sí (ej. `0.892`)

Todo se resuelve en **una sola pasada hacia delante** (~decenas de ms), sin generar texto, sin parsear, con probabilidades honestas.

En este tutorial construimos la intuición desde cero: primero vemos cómo un LLM normal puede "fingir" ser un modelo de decisión (Paso 1), luego cómo un encoder lo hace más eficiente (Paso 2), y finalmente usamos Laya, que está entrenado específicamente para esto (Paso 3).

## Paso 0 — Instalación de dependencias

In [1]:
!pip install -q "torch==2.6.0" --index-url https://download.pytorch.org/whl/cpu
!pip install -q transformers numpy optimum[onnxruntime]  # opcional, para la parte ONNX
!pip install -q laya                  # Paso 3: modelo de decisión estilo JEV

---
## Paso 1 — LLM generativo: porcentajes a la salida

**Idea:** un LLM predice el *siguiente token*. Si forzamos a que el siguiente token sea una etiqueta (`A`, `B`, `C`...), podemos leer el **logit** asignado a cada etiqueta y convertirlos en **probabilidades con softmax**.

Es decir: no pedimos al modelo que *escriba* la respuesta; miramos *dentro* de su cabeza y medimos cuánto "quiere" cada opción.

Usaremos primero un modelo pequeño: `Qwen/Qwen2.5-1.5B-Instruct` (o `HuggingFaceTB/SmolLM2-135M-Instruct` si quieres ir rápido).

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Cargar modelo y tokenizer (usa SmolLM2-135M si tu máquina es modesta)
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

ValueError: Duplicate dispatch rule for <built-in function intern>: already registered in VariableBuilder's id dispatch map

In [ ]:
# 2. Definir el caso de uso: clasificar un ticket en un departamento
choices = {
    "A": "billing and payments",
    "B": "technical support",
    "C": "account access"
}
ticket = "I was charged twice for the same subscription"

choice_lines = "\n".join([f"{key}. {value}" for key, value in choices.items()])

prompt = f"""Question:
Which category matches the ticket?

Allowed labels:
{choice_lines}
Return only the label

Ticket:
{ticket}
Label:"""

print(prompt)

In [ ]:
# 3. Forward pass: obtener logits del ÚLTIMO token (la predicción del "siguiente token")
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    next_token_logits = outputs.logits[0, -1, :]

# 4. Extraer el logit de cada etiqueta candidata
label_logits = {}
for key in choices:
    token_id = tokenizer.encode(key, add_special_tokens=False)[0]
    label_logits[key] = next_token_logits[token_id].item()

# 5. Softmax sobre SOLO los candidatos → porcentajes
candidate_logits = torch.tensor([label_logits[k] for k in choices])
probabilities = torch.softmax(candidate_logits, dim=-1)

print("Logits crudos:")
for key, logit in label_logits.items():
    print(f"  Opción {key} ({choices[key]:<22}): {logit:.4f}")

print("\nProbabilidades (softmax sobre candidatos):")
for key, prob in zip(choices, probabilities):
    print(f"  Opción {key} ({choices[key]:<22}): {prob.item()*100:.1f}%")

**Qué acaba de pasar:**

1. El LLM procesó todo el prompt y, en la última posición, tiene un vector de logits (uno por token del vocabulario, ~50k valores).
2. Nosotros solo miramos los logits de `A`, `B` y `C`, y aplicamos softmax **solo entre esos 3**.
3. Eso nos da una distribución de probabilidad sobre las opciones → el modelo no solo elige, también dice **cuán seguro está**.

**Problemas de este enfoque:**
- Requiere un prompt largo y bien diseñado (formato chat, instrucciones...).
- El modelo generativo está entrenado para *escribir texto*, no para dar probabilidades calibradas (suele estar **sobre-confiado**).
- Latencia proporcional al prompt (pre-fill de todos los tokens).
- Una pregunta a la vez: para N preguntas, N forward passes (o prompts enormes).

→ Paso 2: sustituimos el generador por un **encoder clasificador**.

---
## Paso 2 — Encoder zero-shot (DeBERTa NLI): sin prompts

**Idea:** en lugar de un LLM generativo, usamos un modelo encoder entrenado para **NLI (Natural Language Inference)**: dado un par (premisa, hipótesis), dice si la hipótesis se *implica* (entailment) de la premisa.

Truco de zero-shot: convertimos cada opción en una hipótesis ("This text is about *billing*...") y nos quedamos con el logit de *entailment* de cada una. Softmax entre las opciones → porcentajes.

**Ventajas:** no hay prompt que diseñar, es mucho más rápido (encoder, una pasada por par, pares en batch) y más pequeño.

In [ ]:
import time
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Cargar cross-encoder NLI (clasifica premisa-hipótesis)
model_id_deb = "cross-encoder/nli-deberta-v3-small"
tok_deb = AutoTokenizer.from_pretrained(model_id_deb)
model_deb = AutoModelForSequenceClassification.from_pretrained(model_id_deb)
model_deb.eval()

# 2. Convertir cada opción en una hipótesis → pares (premisa, hipótesis)
pairs = [[ticket, f"This text is about {choices[k]}."] for k in choices]
print(pairs)

In [ ]:
# 3. Una sola pasada en batch para los 3 pares
inputs_deb = tok_deb(pairs, padding=True, truncation=True, return_tensors="pt")

with torch.no_grad():
    logits = model_deb(**inputs_deb).logits  # columnas: [contradiction, entailment, neutral]

# 4. Nos quedamos con el logit de ENTAILMENT (índice 1) para cada opción
label_logits_deb = {k: float(logits[i, 1]) for i, k in enumerate(choices)}

# 5. Softmax entre opciones → porcentajes
cand = np.array([label_logits_deb[k] for k in choices])
exp_l = np.exp(cand - cand.max())
probs_deb = exp_l / exp_l.sum()

for k, p in zip(choices, probs_deb):
    bar = "█" * int(p * 20)
    print(f"  {k} ({choices[k]:<22}): {p*100:5.1f}% | logit entail: {label_logits_deb[k]:6.2f} | {bar}")

In [ ]:
# 6. Medir latencia (promedio de 50 iteraciones)
_ = model_deb(**inputs_deb)  # warmup
latencias = []
for _ in range(50):
    t0 = time.perf_counter()
    with torch.no_grad():
        model_deb(**inputs_deb)
    latencias.append((time.perf_counter() - t0) * 1000)
print(f"Latencia media: {np.mean(latencias):.2f} ms | P95: {np.percentile(latencias, 95):.2f} ms")

**Comparativa rápida (la verás en los resultados):**

| | LLM generativo (Paso 1) | Encoder NLI (Paso 2) |
|---|---|---|
| Prompt | Hay que diseñarlo | No hay prompt, solo pares |
| Latencia | Alta (pre-fill largo) | Baja |
| Calibración | Sobre-confiado | Mejor, pero zero-shot genérico |
| Preguntas por llamada | 1 | 1 (por par) |

Ambos enfoques resuelven **una pregunta tipo `choice`**. Pero un caso real necesita responder **varias preguntas tipadas a la vez** (¿departamento? ¿urgencia? ¿riesgo de churn?). Ahí es donde entran los modelos de decisión entrenados a propósito como **Laya**.

---
## Paso 3 — Laya: un modelo de decisión estilo JEV

[Laya](https://huggingface.co/convaiinnovations/laya) (Convai Innovations, Apache 2.0) es un modelo **no autoregresivo tipo System 1** basado en ModernBERT-large (421M parámetros):

- Le das un **state** (texto, email, ticket o JSON) y un diccionario de **preguntas tipadas**.
- Responde **TODAS las preguntas en UN solo forward pass** (~33 ms), con probabilidades calibradas (entrenado con RLCD).
- **Nunca genera texto**: nada que parsear, nada que alucinar.
- El espacio de respuestas se define **en tiempo de ejecución**: puedes cambiar las opciones sin reentrenar.

Soporta tres tipos de pregunta: `choice` (elegir opción), `score` (puntuación ordinal) y `noul` (sí/no como probabilidad).

Además tiene un `Router` que detecta el idioma (<0.5 ms) y despacha automáticamente al checkpoint inglés o al multilingüe (100+ idiomas).

In [ ]:
from laya import Router

# El Router detecta idioma y usa el checkpoint adecuado (inglés o multilingüe)
router = Router()

state = "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan."
questions = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, system errors",
            "other": "everything else"
        }
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this?",
        "criteria": ["not urgent", "soon", "blocking"]
    },
    "churn_risk": {
        "type": "noul",
        "instructions": "Does the user threaten to cancel or leave?"
    },
}

result = router.predict(state, questions)
print(result)

In [ ]:
# Acceso a las respuestas tipadas
print("Departamento elegido      :", result["answers"]["department"]["choice"])
print("Urgencia (score ordinal)  :", result["answers"]["urgency"])
print("Riesgo de churn (prob.)   :", result["answers"]["churn_risk"]["noul"])
print("Modelo usado por el router:", result["routing"]["model"])

In [ ]:
# El Router detecta idioma: probemos en español (despacha al checkpoint multilingüe)
state_es = "Me han cobrado dos veces la cuota de marzo. Quiero la devolución hoy o daré de baja mi suscripción."
result_es = router.predict(state_es, questions)

print("Modelo usado  :", result_es["routing"]["model"])
print("Departamento  :", result_es["answers"]["department"]["choice"])
print("Riesgo churn  :", result_es["answers"]["churn_risk"]["noul"])

In [ ]:
# Latencia: todas las preguntas en UNA pasada
import time
import numpy as np

router.predict(state, questions)  # warmup
latencias_laya = []
for _ in range(50):
    t0 = time.perf_counter()
    router.predict(state, questions)
    latencias_laya.append((time.perf_counter() - t0) * 1000)

print(f"Laya (3 preguntas tipadas): media {np.mean(latencias_laya):.2f} ms | P95 {np.percentile(latencias_laya, 95):.2f} ms")
print("(Compara con los pasos 1 y 2: allí, UNA pregunta choice ya costaba más)")

### Modo modelo único (sin Router)

Si no quieres el enrutado por idioma, puedes cargar el modelo directamente:

In [ ]:
import laya

agent = laya.load("convaiinnovations/laya")  # checkpoint inglés (~808 MB)
result2 = agent.predict(state, questions)    # TODAS las preguntas en un forward pass
print(result2["answers"])

### Límites (del model card de Laya)

- Los checkpoints base son **casi aleatorios zero-shot** en typed-decisions (0.362); el 0.766 viene del checkpoint fine-tuned (`laya-typed-decisions`).
- Con **muchas opciones** (>20) degrada: divide jerárquicamente o sube `head_max_len`.
- `noul` puede dejarse llevar por sus etiquetas `false:`/`true:` — valida en tus datos o usa un `choice` de dos opciones.
- Viene **sobre-confiado**: recalibra (temperature scaling) una vez por (tipo, nº opciones) sobre tus datos (ECE 0.466 → 0.081 tras calibrar).
- El checkpoint raíz es solo inglés; usa `laya-multilingual` para el resto de idiomas (el Router lo hace solo).

Puedes comparar modelos de decisión en el leaderboard: **Jev Decision Index** → https://huggingface.co/spaces/multimodalart/jev-decision-index (ahí verás a Jev, Laya, Bosun, Decider, GLiNER2.5, etc. puntuados en benchmarks de decisiones tipadas).

---
## Resumen del tutorial

| Paso | Modelo | Cómo decide | Preguntas/llamada | Cuándo usarlo |
|------|--------|-------------|-------------------|----------------|
| 1 | LLM generativo (SmolLM2/Qwen) | Logits del siguiente token + softmax entre etiquetas | 1 | Prototipado rápido, flexibilidad total |
| 2 | Encoder NLI (DeBERTa) | Logit de entailment por hipótesis | 1 (en batch) | Zero-shot sin prompt, latencia baja |
| 3 | Laya (System 1) | Forward pass entrenado para decisiones tipadas | **Todas a la vez** | Producción: routers, triaje, guardrails |

**Idea central:** un modelo de decisión no *escribe* la respuesta — la **puntúa**. Eso lo hace rápido, determinista y con probabilidades que puedes usar para tomar decisiones automáticas (ej. "si churn_risk > 0.8 → escalar a retención").